<a href="https://colab.research.google.com/github/VictorNevola/ml-study/blob/main/ml_12_regress%C3%A3o_linear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mp
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [4]:
df = pd.read_csv("diabetes_en.csv")
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# logistic regression needs scaling (gradient-based + regularized by default)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)

print(f"Accuracy: {model.score(X_test, y_test):.2%}")

Accuracy: 71.43%


In [5]:

probabilities = model.predict_proba(X_test)[:, 1]

preview = pd.DataFrame({
    "probability_of_diabetes": probabilities[:8].round(3),
    "prediction": model.predict(X_test)[:8],
    "actual": y_test.values[:8],
})
preview

,probability_of_diabetes,prediction,actual
0,0.617,1,0
1,0.113,0,0
2,0.273,0,0
3,0.287,0,1
4,0.004,0,0
5,0.189,0,0
6,0.468,0,1
7,0.917,1,1


In [7]:
coefs = pd.Series(
    model.named_steps["classifier"].coef_[0],
    index=X.columns
).sort_values(ascending=False)
coefs.round(3)

,0
glucose,1.144
bmi,0.714
pregnancies,0.373
diabetes_pedigree,0.256
age,0.184
skin_thickness,0.067
insulin,-0.127
blood_pressure,-0.198


In [8]:

from sklearn.metrics import precision_score, recall_score

print(f"{'threshold':>10} | {'precision':>9} | {'recall':>7}")
print("-" * 32)
for threshold in [0.7, 0.5, 0.3, 0.2]:
    preds = (probabilities >= threshold).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds)
    print(f"{threshold:>10.1f} | {p:>9.2%} | {r:>7.2%}")


 threshold | precision |  recall
--------------------------------
       0.7 |    71.43% |  37.04%
       0.5 |    60.87% |  51.85%
       0.3 |    59.46% |  81.48%
       0.2 |    53.93% |  88.89%
